# Loan Default Prediction

A probability-of-default (PD) model on the LendingClub dataset (2007-2018Q4).
The notebook keeps the exploratory analysis, the plots and the narrative; the
processing, training and evaluation steps live in the importable modules under
`src/` (see issue #4).

**How to run it:** install `requirements.txt`, place the raw file
`accepted_2007_to_2018Q4.csv` in `02_data/raw/`, then run the notebook from top
to bottom ("Restart & Run All"). Non-interactively that is:

```
jupyter nbconvert --to notebook --execute --inplace 01_notebooks/prediction.ipynb
```

The raw CSV (~1.6 GB) is not part of this repository; it comes from the
LendingClub dataset on Kaggle.

In [ ]:
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# The notebook lives in 01_notebooks/, the src package in the project root.
sys.path.insert(0, "..")

from src.data_processing import (
    add_issue_year,
    create_target,
    drop_leakage_columns,
    filter_issue_years,
    load_raw_data,
    rm_nas,
)
from src.evaluate import evaluate_model
from src.features import engineer_features
from src.train import save_model, split_and_scale, train_random_forest

sns.set_style("whitegrid")
warnings.filterwarnings('ignore')

## 1. Load Data

The raw file holds ~2.2M rows, so loading and processing it takes a few minutes
and a fair amount of memory. For a quick pass over the notebook, pass `nrows` to
`load_raw_data` to read only a sample.

In [ ]:
print("Loading the full dataset... This may take a few minutes.")
df = load_raw_data("../02_data/raw/accepted_2007_to_2018Q4.csv")

# For quick testing on a smaller sample:
# df = load_raw_data("../02_data/raw/accepted_2007_to_2018Q4.csv", nrows=100000)

print(f"Shape of the dataset: {df.shape}")

## 2. Exploratory Data Analysis

A first look at size, distributions and the correlation structure of the data.
The year plot motivates the one restriction this section already applies: the
filter to the 2015-2018 window a few cells below.

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
df['grade'].value_counts()

In [ ]:
# Loan amounts are the exposure at risk, so their distribution sets the scale
# for everything that follows.
plt.figure(figsize=(10, 6))
sns.histplot(df['loan_amnt'], bins=50, kde=True)
plt.title('Distribution of Loan Amounts')
plt.show()

In [ ]:
# 'issue_year' is derived from 'issue_d'; it is used for the plot below and
# for the year filter in the next cell.
df = add_issue_year(df)

plt.figure(figsize=(12, 7))
sns.countplot(x='issue_year', data=df, palette='viridis')
plt.title('Number of Loans Issued Per Year', fontsize=16)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Count of Loans', fontsize=12)
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Restrict the data to 2015-2018. The plot above shows that LendingClub's
# volume grew by orders of magnitude over the years, and lending standards
# changed with it. A recent, homogeneous window is closer to the population a
# scoring model would be applied to than the full 2007-2018 history.
print(f"Shape of the original DataFrame: {df.shape}")

df = filter_issue_years(df, start_year=2015, end_year=2018)

print(f"Shape of the new filtered DataFrame: {df.shape}")
print("\nYearly counts in the new filtered DataFrame:")
print(df['issue_year'].value_counts().sort_index())

In [ ]:
# Select only numeric columns for correlation
numeric_df = df.select_dtypes(include=np.number)

plt.figure(figsize=(12, 10))
sns.heatmap(numeric_df.corr(), cmap='coolwarm')
plt.title('Correlation Matrix of Numerical Features')
plt.show()

## 3. Data Cleaning & Preprocessing

Four cleaning steps, in this order:

1. Keep only loans with a known outcome and derive the binary target from
   `loan_status`.
2. Drop columns that are mostly empty.
3. Drop columns that are irrelevant (identifiers, free text) or that leak the
   outcome.
4. Drop the rows that still have missing values.

The order matters: the target is defined first so that the later steps can be
judged against it, and the leakage drop runs after the missing-value column
drop (step 2) so that it also catches columns the 40% threshold happened to
keep.

In [ ]:
# Only 'Fully Paid' and 'Charged Off' loans have a final outcome. Loans that
# are still running would otherwise be labelled as non-defaults although their
# outcome is simply not known yet.
df = create_target(df)

print("Target variable 'target' created.")
print(df['target'].value_counts(normalize=True) * 100)

In [ ]:
# Drop columns with more than 40% missing values. Imputing a column that is
# mostly empty invents more data than it recovers; 40% is the point at which
# the LendingClub columns split cleanly into "generally populated" and
# "only filled for a special case" (hardship, settlement, joint applications).
df = rm_nas(df, threshold=0.4)
print(f"Shape after dropping columns with >40% NaNs: {df.shape}")

### Data leakage: post-outcome columns are removed

The first version of this notebook reported a ROC AUC of **0.9999**. On the
LendingClub dataset such a value is not a success but a known warning sign for
data leakage: several columns are only filled *after* the loan has already
defaulted or been repaid (payments received, outstanding principal, recoveries,
the continuously updated `last_fico_range_*` score, hardship and settlement
fields). Used as features, they tell the model the outcome it is supposed to
predict.

The cell below therefore removes all post-outcome columns in addition to the
irrelevant ones. What remains is the information that is available at
**origination** — exactly the information a real PD scoring model may use. The
resulting AUC is much lower and, for a credit default model on this kind of
data, the credible one (typically in the range of about 0.65-0.75).

In [ ]:
# Remove irrelevant explanatory variables (identifiers, free text, date
# columns) together with every post-outcome column described above. The
# concrete column lists live in src/data_processing.py.
df = drop_leakage_columns(df)
print(f"Shape after dropping irrelevant and post-outcome columns: {df.shape}")

In [ ]:
# After the column drops the remaining NaNs affect only a small share of rows,
# so dropping those rows is cheaper than imputing them.
df.dropna(inplace=True)
print(f"Shape after dropping all remaining NaN rows: {df.shape}")

## 4. Feature Engineering

Everything the model sees has to be numeric. `term` and `emp_length` are stored
as text but are genuinely ordinal, so they are parsed into numbers rather than
one-hot encoded. The remaining categorical columns get one-hot encoding with
`drop_first=True` to avoid the redundant reference category.

In [ ]:
df = engineer_features(df)

print("Feature engineering complete.")
print(f"Final shape of data for modeling: {df.shape}")

## 5. Model Training & Evaluation

A Random Forest is used as the first model: it handles the mix of numeric and
one-hot encoded features without further preprocessing, copes with non-linear
relationships and is robust against the outliers that credit data is full of.
The interpretable counterpart that a bank would expect alongside it — logistic
regression on WOE-binned features — is the subject of a separate issue.

Defaults are the minority class (roughly 20% of completed loans), so
`class_weight="balanced"` is used instead of resampling, and the train/test
split is stratified so both sets carry the same default rate.

In [ ]:
# The scaler inside split_and_scale is fitted on the training set only, so no
# information from the test set leaks into training.
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test, scaler = split_and_scale(
    X, y, test_size=0.3, random_state=42
)

print("Data split into training and test sets, and features scaled.")

In [ ]:
print("Training Random Forest model... This might be slow on the full dataset.")

model = train_random_forest(X_train, y_train)
save_model(model, "../04_models/random_forest.joblib")

print("Model training complete, model saved to 04_models/random_forest.joblib.")

In [ ]:
# ROC AUC is the headline metric because it is threshold-independent; the
# classification report adds precision and recall at the default 0.5 cutoff.
scores = evaluate_model(model, X_test, y_test)

print(f"\nROC AUC Score: {scores['roc_auc']:.4f}")
print("\nClassification Report:")
print(scores['report'])

### Comments on Findings

The model above only uses features that are known at origination. The
corrected run on the full dataset is still outstanding (the raw CSV is not
part of this repository), so the concrete metrics are not filled in here yet.

What the corrected model must show is an AUC far below the 0.9999 of the first
version — that drop is the point: the earlier value came from post-outcome
columns leaking the target (see the section on data leakage above), not from
predictive power. A value in the region of 0.65-0.75 is what a realistic PD
model on LendingClub data achieves and is the number that can be defended in a
credit risk context.

For the business interpretation, recall on the 'Charged Off (1)' class is the
relevant quantity: it shows how many of the actually defaulting loans the model
flags. Precision and recall are now clearly traded off against each other, so
the decision threshold has to be chosen according to the cost of a missed
default versus a rejected good customer.